# **Notebook 4: Evaluation**
**Goal:** Compute precision, recall, F1, and mean angle error.

This notebook produces the exact numbers you fill into the submission form.

In [ ]:
import os, cv2, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
# Load ground truth and predictions
gt   = pd.read_csv('/content/dataset/annotations.csv')
pred = pd.read_csv('/content/final_predictions.csv')
print(f"GT rows  : {len(gt)}")
print(f"Pred rows: {len(pred)}")

In [ ]:
# ── Core matching function ────────────────────────────────────────────────────
def circular_angle_error(a, b):
    """Smallest angular difference between two angles (handles 0/360 wrap)."""
    d = abs(a - b) % 360
    return min(d, 360 - d)


def evaluate(gt_df, pred_df, dist_thresh=30.0):
    images = gt_df['image'].unique()
    tp = fp = fn = 0
    angle_errors = []
    for img_name in images:
        gtr  = gt_df[gt_df['image']    == img_name].reset_index(drop=True)
        prdr = pred_df[pred_df['image'] == img_name].reset_index(drop=True)
        ng, np_ = len(gtr), len(prdr)
        if np_ == 0: fn += ng; continue
        if ng  == 0: fp += np_; continue
        cost = np.linalg.norm(
            gtr[['center_x','center_y']].values[:,None,:] -
            prdr[['center_x','center_y']].values[None,:,:], axis=2)
        ri, ci = linear_sum_assignment(cost)
        mg, mp = set(), set()
        for r, c in zip(ri, ci):
            if cost[r,c] <= dist_thresh:
                tp+=1; mg.add(r); mp.add(c)
                d = abs(prdr.iloc[c]['angle_deg']-gtr.iloc[r]['angle_deg']) % 360
                angle_errors.append(min(d, 360-d))
        fn += ng-len(mg); fp += np_-len(mp)
    prec = tp/(tp+fp) if tp+fp else 0
    rec  = tp/(tp+fn) if tp+fn else 0
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0
    return {'precision': round(prec,4), 'recall': round(rec,4), 'f1': round(f1,4),
            'mean_angle_err':   round(np.mean(angle_errors),2)   if angle_errors else None,
            'median_angle_err': round(np.median(angle_errors),2) if angle_errors else None,
            'tp':tp,'fp':fp,'fn':fn,'n_angle_evals':len(angle_errors),
            'dist_threshold_px': dist_thresh}

metrics = evaluate(gt_val, final_df, DIST_THRESH)
metrics['note']             = f'Val split only ({len(val_set)} images, random_state=42)'
metrics['angle_method']     = 'contour_deviation_largest_component'
metrics['detection_method'] = 'yolov8n_finetuned_50epochs'

print('=' * 52)
print(f"  TP/FP/FN       : {metrics['tp']}/{metrics['fp']}/{metrics['fn']}")
print(f"  Precision      : {metrics['precision']:.4f}")
print(f"  Recall         : {metrics['recall']:.4f}")
print(f"  F1             : {metrics['f1']:.4f}")
print(f"  Mean angle err : {metrics['mean_angle_err']}°")
print(f"  Median ang err : {metrics['median_angle_err']}°")
print('=' * 52)

with open(METRICS_JSON, 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


In [ ]:
# ── 6. Per-image breakdown ─────────────────────────────────────────────────────
per_img['errors'] = per_img['fp'] + per_img['fn']
print("Per-image (sorted worst first):")
display(per_img.sort_values('errors', ascending=False).reset_index(drop=True))

In [ ]:
# ── 8. Qualitative overlay ─────────────────────────────────────────────────────
show_imgs = list(val_set)[:6]
ncols = 3; nrows = math.ceil(len(show_imgs) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*6, nrows*5))
axes = np.array(axes).flatten()

for i, img_name in enumerate(show_imgs):
    img = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, img_name)), cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)

    gtr  = gt_val[gt_val['image'] == img_name].reset_index(drop=True)
    prdr = final_df[final_df['image'] == img_name].reset_index(drop=True)

    # Compute matched predictions for TP/FP colouring
    matched_p = set()
    if len(gtr) > 0 and len(prdr) > 0:
        cost = np.linalg.norm(
            gtr[['center_x','center_y']].values[:,None,:] -
            prdr[['center_x','center_y']].values[None,:,:], axis=2)
        ri, ci = linear_sum_assignment(cost)
        matched_p = {c for r,c in zip(ri,ci) if cost[r,c] <= DIST_THRESH}

    # GT: green cross + angle arrow
    for _, r in gtr.iterrows():
        axes[i].plot(r['center_x'], r['center_y'], 'g+', ms=14, mew=2)
        rad = math.radians(r['angle_deg'])
        axes[i].annotate('', xy=(r['center_x']+28*math.cos(rad),
                                  r['center_y']-28*math.sin(rad)),
                         xytext=(r['center_x'], r['center_y']),
                         arrowprops=dict(arrowstyle='->', color='lime', lw=2))

    # Predictions: cyan=TP, red=FP
    for j, (_, r) in enumerate(prdr.iterrows()):
        col = 'cyan' if j in matched_p else 'red'
        axes[i].plot(r['center_x'], r['center_y'], 'x', color=col, ms=10, mew=2)
        rad = math.radians(r['angle_deg'])
        axes[i].annotate('', xy=(r['center_x']+22*math.cos(rad),
                                  r['center_y']-22*math.sin(rad)),
                         xytext=(r['center_x'], r['center_y']),
                         arrowprops=dict(arrowstyle='->', color=col, lw=1.5))

    axes[i].set_title(img_name, fontsize=8); axes[i].axis('off')

for j in range(i+1, len(axes)): axes[j].axis('off')

legend = [Line2D([0],[0],marker='+', color='lime', label='GT', ms=12, lw=0),
          Line2D([0],[0],marker='x', color='cyan', label='Pred TP', ms=10, lw=0),
          Line2D([0],[0],marker='x', color='red',  label='Pred FP', ms=10, lw=0)]
fig.legend(handles=legend, loc='lower center', ncol=3, fontsize=10)
plt.suptitle('Qualitative overlay — val split', fontsize=13)
plt.tight_layout(rect=[0,0.04,1,1])
plt.savefig('/content/qualitative_overlay.png', dpi=150)
plt.show()